# AdaptiveSats Regime-Based BTC Accumulation Strategy

**Author:** Raghav Gupta

This notebook implements and evaluates multiple regime-based Bitcoin accumulation strategies against a uniform DCA benchmark.

## Main terms used in the notebook

- **DCA**: Uniform dollar-cost averaging. Every day in a 365-day window gets the same allocation weight.
- **SPD / sats per dollar**: Number of satoshis accumulated per USD spent. Higher is better.
- **Window**: A fixed 365-day evaluation period. Each window receives the same budget.
- **Regime**: A market condition label based on BTC trend, MVRV valuation, and realized-cap versus market-cap growth.
- **Candidate strategy**: A strategy that can be selected inside a regime, such as MVRV, Momentum, SMA, or Composite.
- **Composite strategy**: A weighted combination of on-chain signals using causal rolling z-scores.
- **Causal rolling z-score**: A z-score calculated using only current and past values, not future values.
- **MAX_DCA_MULTIPLE**: The cap that prevents any single day from receiving more than a fixed multiple of normal DCA allocation.


## Cell 1: Imports, data preparation check, and configuration

Set up the notebook: imports, StackSats dataset checks, train/test dates, budget, lookbacks, allocation cap, composite signal list, and candidate strategy names.

In [1]:
# Cell 1: Imports, data preparation check, and configuration
# ============================================================
# This cell imports required libraries, checks whether the prepared
# StackSats BTC analytics dataset exists, prepares it if missing,
# and defines all strategy settings.

import polars as pl
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import subprocess

from stacksats.runner.core import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.mvrv.core import MVRVStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy

# Optimization import used by the composite strategy.
# If this fails, install scipy in your environment:
# pip install scipy
try:
    from scipy.optimize import minimize
except ImportError as exc:
    raise ImportError(
        "scipy is not installed. Install it using: pip install scipy "
        "or add `scipy` to your environment.yml."
    ) from exc

# ============================================================
# StackSats prepared dataset check
# ============================================================
# pip install stacksats installs the package, but it does not automatically
# create ~/.stacksats/data/bitcoin_analytics.parquet.
# This block prepares the file if it is missing.

btc_path = Path.home() / ".stacksats" / "data" / "bitcoin_analytics.parquet"

# IMPORTANT:
# Update this path if your brk_metrics.parquet is in a different location.
# If this notebook is inside the notebooks/ folder and data/ is at repo root,
# then ../data/brk_metrics.parquet is usually correct.
raw_brk_path = Path("../data/brk_metrics.parquet")

# Processed long-format BRK metrics used for the composite strategy.
# Expected format: day_utc | metric | value
processed_brk_metrics_path = Path("../data/processed/brk_metrics.parquet")

if not btc_path.exists():
    print(f"Prepared dataset not found at: {btc_path}")
    print("Preparing StackSats analytics dataset...")

    if not raw_brk_path.exists():
        raise FileNotFoundError(
            f"Raw BRK metrics file not found at: {raw_brk_path}. "
            "Please update raw_brk_path to the correct location of brk_metrics.parquet."
        )

    subprocess.run(
        [
            "stacksats",
            "data",
            "prepare",
            "--source",
            str(raw_brk_path),
        ],
        check=True,
    )

if not btc_path.exists():
    raise FileNotFoundError(
        f"Failed to create prepared dataset at {btc_path}."
    )

print(f"Using prepared dataset: {btc_path}")


# ============================================================
# Strategy configuration
# ============================================================

# Budget used per 365-day window.
TOTAL_BUDGET_USD = 1000.0

# Train and test periods.
TRAIN_START = "2018-01-01"
TRAIN_END = "2023-12-31"
TEST_START = "2024-01-01"
TEST_END = "2025-12-31"

# Each evaluation window is 365 days.
WINDOW_SIZE = 365

# StackSats-style exponential decay factor for percentile aggregation.
# 0.9 means each older window receives 90% of the weight of the next newer window.
EXP_DECAY_FACTOR = 0.90


# Lookbacks used for momentum, SMA, drawdown, and regime classification.
MOMENTUM_LOOKBACK = 45
SMA_LOOKBACK = 180
DRAWDOWN_LOOKBACK = 180
REGIME_LOOKBACK = 180

# Unique lookback values used to create rolling features.
LOOKBACK_DAYS = sorted({
    MOMENTUM_LOOKBACK,
    SMA_LOOKBACK,
    REGIME_LOOKBACK,
})

# Small floor to avoid zero or negative allocation signals.
SIGNAL_FLOOR = 1e-8

# Maximum final daily allocation relative to uniform DCA.
# Example: for a 365-day window, DCA weight = 1/365 = 0.0027397.
# With MAX_DCA_MULTIPLE = 15, no day can receive more than:
# 15 * 1/365 = 0.041096, or about 4.11% of the window budget.
# Fixed final daily allocation cap.
# No day can receive more than MAX_DCA_MULTIPLE times the uniform DCA allocation.
MAX_DCA_MULTIPLE = 15.0

# Minimum number of days required for a regime to be evaluated.
MIN_REGIME_DAYS = 20

# Tolerance used to decide whether a result is better, worse, or tied.
STATUS_TOLERANCE_PCT = 1e-6

# Composite on-chain strategy settings.
# This strategy uses extra BRK metrics from:
# ../data/processed/brk_metrics.parquet
#
# The file is expected to be long-format:
# day_utc | metric | value
#
# Nelder-Mead starts from equal weights and optimizes signal weights + gamma
# using the training period only.
COMPOSITE_ROLLING_WINDOW = 365
COMPOSITE_SIGNAL_STRENGTH = 1.25
COMPOSITE_GAMMA = 1.00

COMPOSITE_SIGNAL_COLS = [
    "mvrv",
    "sopr_7d_ema",
    "reserve_risk",
    "greed_index",
    "puell_multiple",
    "lth_nupl",
    "sell_side_risk_ratio_7d_ema",
    "net_unrealized_pnl_rel_to_market_cap",
]

COMPOSITE_Z_COLS = [f"z_{col}" for col in COMPOSITE_SIGNAL_COLS]

COMPOSITE_SIGNAL_WEIGHTS = {
    col: 1.0 / len(COMPOSITE_SIGNAL_COLS)
    for col in COMPOSITE_SIGNAL_COLS
}

# Nelder-Mead optimization settings for the composite strategy.
COMPOSITE_OPT_MAXITER = 600
COMPOSITE_OPT_MAXFEV = 1200

# Gamma controls how aggressively positive cheapness is amplified.
# It is clipped during optimization to reduce overfitting.
COMPOSITE_GAMMA_MIN = 0.25
COMPOSITE_GAMMA_MAX = 3

# Fallback strategy used if a regime appears in test but was not seen in training.
FALLBACK_STRATEGY =  "composite_weight"

# Candidate strategies used in regime selection.
# DCA is benchmark only and is not selected as a candidate here.
# ML target-based strategy has been removed to avoid train/test label-boundary leakage.
# Standalone StackSats MVRV, Momentum, SMA, and Composite remain as candidates.
CANDIDATE_COLS = [
    "stacksats_mvrv_weight",
    "stacksats_momentum_weight",
    f"sma_{SMA_LOOKBACK}d_weight",
    "composite_weight",
]


# ============================================================


Using prepared dataset: C:\Users\ragha\.stacksats\data\bitcoin_analytics.parquet


## Cell 2: Helper functions

Reusable utility functions for labeling results, normalizing weights, computing rolling z-scores, calculating exponential-decay percentiles, and summarizing sats-per-dollar performance.

In [2]:
# Cell 2: Helper functions
# ============================================================
# This cell defines general helper functions used across the notebook.
# These functions are reused for labeling results, formatting chart text,
# trimming complete windows, normalizing weights, and summarizing results.

def get_status_from_pct_diff(pct_diff, tolerance=STATUS_TOLERANCE_PCT):
    """
    Convert percentage improvement into a simple status label.

    Parameters
    ----------
    pct_diff : float
        Percentage difference of strategy performance versus DCA.
    tolerance : float
        Small threshold used to avoid classifying tiny numerical differences
        as meaningful wins or losses.

    Returns
    -------
    str
        'better' if strategy beats DCA, 'worse' if it underperforms,
        and 'tie' if the difference is within tolerance.
    """
    if pct_diff > tolerance:
        return "better"
    if pct_diff < -tolerance:
        return "worse"
    return "tie"


def format_arrow_text(extra_spd, improvement_pct):
    """
    Create chart annotation text for positive or negative SPD improvement.

    Parameters
    ----------
    extra_spd : float
        Extra sats per dollar compared with DCA.
    improvement_pct : float
        Percentage improvement compared with DCA.

    Returns
    -------
    tuple[str, str]
        Formatted text and color name.
    """
    if extra_spd >= 0:
        return f"▲ +{extra_spd:,.2f} sats/$ ({improvement_pct:+.2f}%)", "green"
    return f"▼ {extra_spd:,.2f} sats/$ ({improvement_pct:+.2f}%)", "red"


def trim_full_windows(df: pd.DataFrame, window_size: int = WINDOW_SIZE):
    """
    Keep only complete fixed-length windows.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe sorted by date.
    window_size : int
        Number of rows per evaluation window.

    Returns
    -------
    tuple[pd.DataFrame, int, int]
        Trimmed dataframe, number of complete windows, and number of dropped rows.
    """
    n_full = len(df) // window_size
    n_eval = n_full * window_size
    remainder = len(df) - n_eval
    return df.iloc[:n_eval].copy(), n_full, remainder


def build_simple_normalized_weights(signal_multiplier, signal_floor=SIGNAL_FLOOR):
    """
    Convert signal multipliers into normalized daily allocation weights.

    Parameters
    ----------
    signal_multiplier : array-like
        Raw signal strength or multiplier values.
    signal_floor : float
        Minimum allowed signal value to prevent zero or negative weights.

    Returns
    -------
    np.ndarray
        Daily allocation weights that sum to 1.
    """
    signal_multiplier = np.asarray(signal_multiplier, dtype=float)

    if len(signal_multiplier) == 0:
        raise ValueError("Empty signal array.")

    clean_signal = np.nan_to_num(
        signal_multiplier,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    clean_signal = np.maximum(clean_signal, signal_floor)

    if clean_signal.sum() <= 0:
        return np.full(len(clean_signal), 1.0 / len(clean_signal))

    return clean_signal / clean_signal.sum()


def rolling_zscore(series: pd.Series, window: int = COMPOSITE_ROLLING_WINDOW):
    """
    Compute a causal rolling z-score.

    Each row uses only current/trailing historical values.
    No future values are used.
    """
    rolling_mean = series.rolling(window=window, min_periods=120).mean()
    rolling_std = series.rolling(window=window, min_periods=120).std()
    return (series - rolling_mean) / (rolling_std + 1e-8)


def compute_stack_sats_exp_decay_average(
    percentile_values,
    decay_factor=EXP_DECAY_FACTOR,
):
    """
    Calculate StackSats-style exponentially decayed average percentile.

    Parameters
    ----------
    percentile_values : array-like
        Chronologically ordered percentile values.
        Older values should come first and newer values should come last.
    decay_factor : float
        Exponential decay factor.

    Returns
    -------
    float
        Exponentially decayed average percentile.

        exp_weights = 0.9 ** np.arange(N - 1, -1, -1)
        exp_weights /= exp_weights.sum()
        exp_avg_pct = (percentile_values * exp_weights).sum()

    This gives the newest window the largest weight and older windows
    gradually lower weights.
    """
    values = np.asarray(percentile_values, dtype=float)

    if len(values) == 0:
        return np.nan

    values = np.nan_to_num(
        values,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    n = len(values)

    exp_weights = decay_factor ** np.arange(n - 1, -1, -1)
    exp_weights = exp_weights / exp_weights.sum()

    return float((values * exp_weights).sum())


def add_spd_percentiles_to_window_summary(window_summary_df):
    """
    Add dynamic and uniform SPD percentile columns.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary with strategy_spd and dca_spd.

    Returns
    -------
    pd.DataFrame
        Window summary with:
        - dynamic_percentile
        - uniform_percentile

    Notes
    -----
    Here, percentiles are calculated across the available 365-day windows
    in the current period.
    """
    out = window_summary_df.copy().sort_values("start_date").reset_index(drop=True)

    out["dynamic_percentile"] = (
        out["strategy_spd"]
        .rank(method="average", pct=True)
        * 100.0
    )

    out["uniform_percentile"] = (
        out["dca_spd"]
        .rank(method="average", pct=True)
        * 100.0
    )

    return out


def calculate_stack_sats_exp_decay_metrics(window_summary_df):
    """
    Calculate exp-decay percentile metrics for a window summary.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary with dynamic_percentile and uniform_percentile.

    Returns
    -------
    dict
        Dictionary with exp_decay_percentile and uniform_exp_decay_percentile.
    """
    ordered = window_summary_df.copy().sort_values("start_date").reset_index(drop=True)

    exp_decay_percentile = compute_stack_sats_exp_decay_average(
        ordered["dynamic_percentile"].to_numpy(),
        decay_factor=EXP_DECAY_FACTOR,
    )

    uniform_exp_decay_percentile = compute_stack_sats_exp_decay_average(
        ordered["uniform_percentile"].to_numpy(),
        decay_factor=EXP_DECAY_FACTOR,
    )

    return {
        "exp_decay_percentile": exp_decay_percentile,
        "uniform_exp_decay_percentile": uniform_exp_decay_percentile,
    }




def summarize_spd_like_composite(window_summary_df):
    """
    Summarize performance across all 365-day windows.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary dataframe containing strategy_sats, dca_sats,
        strategy_spd, dca_spd, and result columns.

    Returns
    -------
    dict
        Summary metrics including total sats, SPD sums, improvement percentage,
        wins, losses, ties, and win rate.
    """
    n_windows = len(window_summary_df)

    strategy_spd_sum = window_summary_df["strategy_spd"].sum()
    dca_spd_sum = window_summary_df["dca_spd"].sum()
    extra_spd_sum = strategy_spd_sum - dca_spd_sum

    spd_ratio = strategy_spd_sum / dca_spd_sum
    improvement_pct = (spd_ratio - 1.0) * 100.0

    strategy_sats = window_summary_df["strategy_sats"].sum()
    dca_sats = window_summary_df["dca_sats"].sum()
    extra_sats = strategy_sats - dca_sats

    wins = int((window_summary_df["result"] == "better").sum())
    losses = int((window_summary_df["result"] == "worse").sum())
    ties = int((window_summary_df["result"] == "tie").sum())

    win_rate_pct = wins / n_windows * 100.0 if n_windows > 0 else 0.0

    if {"dynamic_percentile", "uniform_percentile"}.issubset(window_summary_df.columns):
        exp_decay_metrics = calculate_stack_sats_exp_decay_metrics(window_summary_df)
    else:
        exp_decay_metrics = {
            "exp_decay_percentile": np.nan,
            "uniform_exp_decay_percentile": np.nan,
        }

    return {
        "n_windows": n_windows,
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "win_rate_pct": win_rate_pct,
        "exp_decay_percentile": exp_decay_metrics.get("exp_decay_percentile", np.nan),
        "uniform_exp_decay_percentile": exp_decay_metrics.get("uniform_exp_decay_percentile", np.nan),

        "strategy_sats": strategy_sats,
        "dca_sats": dca_sats,
        "extra_sats_vs_dca": extra_sats,

        "strategy_spd_sum": strategy_spd_sum,
        "dca_spd_sum": dca_spd_sum,
        "extra_spd_sum_vs_dca": extra_spd_sum,

        "strategy_spd_avg": strategy_spd_sum / n_windows,
        "dca_spd_avg": dca_spd_sum / n_windows,
        "extra_spd_avg_vs_dca": extra_spd_sum / n_windows,

        "spd_ratio": spd_ratio,
        "improvement_pct": improvement_pct,
    }


def build_log_tick_values_and_text():
    """
    Build custom y-axis tick values and labels for the BTC log-price chart.

    Returns
    -------
    tuple[list[int], list[str]]
        Tick values and corresponding display labels.
    """
    tickvals = [
        3000, 4000, 5000, 6000, 7000, 8000, 9000,
        10000,
        20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000,
        100000,
    ]

    ticktext = [
        "3", "4", "5", "6", "7", "8", "9",
        "10k",
        "2", "3", "4", "5", "6", "7", "8", "9",
        "100k",
    ]

    return tickvals, ticktext


# ============================================================


## Cell 3: Load BTC data

Load the prepared Bitcoin analytics dataset and verify that all required price and on-chain columns exist before strategy logic runs.

In [3]:
# Cell 3: Load BTC data
# ============================================================
# This cell loads the prepared Bitcoin analytics parquet file.
# It checks that all required columns exist before moving forward.

if not btc_path.exists():
    raise FileNotFoundError(f"Could not find: {btc_path}")

btc_df = (
    pl.read_parquet(btc_path)
    .with_columns(pl.col("date").cast(pl.Datetime))
    .sort("date")
)

required_cols = [
    "date",
    "price_usd",
    "mvrv",
    "adjusted_sopr",
    "adjusted_sopr_7d_ema",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

missing_cols = [col for col in required_cols if col not in btc_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns from bitcoin_analytics.parquet: {missing_cols}")

print("Loaded BTC rows:", btc_df.height)

display(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date"),
    )
)


# ============================================================


Loaded BTC rows: 5689


min_date,max_date
datetime[μs],datetime[μs]
2010-08-16 00:00:00,2026-03-13 00:00:00


## Cell 4: StackSats strategy export setup

Initialize the StackSats runner and define a cached exporter for built-in MVRV and Momentum daily allocation weights.

In [4]:
# Cell 4: StackSats strategy export setup
# ============================================================
# This cell sets up the StackSats strategy runner.
# It exports daily weights from built-in StackSats MVRV and Momentum strategies.

runner = StrategyRunner()

stacksats_strategy_objects = {
    "stacksats_mvrv_weight": MVRVStrategy(),
    "stacksats_momentum_weight": MomentumStrategy(),
}

# Cache prevents recomputing weights for the same strategy/window repeatedly.
_export_cache = {}


def export_stacksats_weights_for_window(
    strategy_key: str,
    window_df: pd.DataFrame,
    full_btc_df: pl.DataFrame,
):
    """
    Export StackSats strategy weights for one 365-day window.

    Parameters
    ----------
    strategy_key : str
        Key identifying which StackSats strategy to run.
    window_df : pd.DataFrame
        Current 365-day window dataframe.
    full_btc_df : pl.DataFrame
        Full BTC analytics dataframe in Polars format.

    Returns
    -------
    np.ndarray
        Normalized daily weights for the selected StackSats strategy.
    """
    window_start = pd.to_datetime(window_df["date"].min()).strftime("%Y-%m-%d")
    window_end = pd.to_datetime(window_df["date"].max()).strftime("%Y-%m-%d")

    cache_key = (strategy_key, window_start, window_end)

    if cache_key in _export_cache:
        return _export_cache[cache_key].copy()

    config = ExportConfig(
        range_start=window_start,
        range_end=window_end,
    )

    window_btc_df = (
        full_btc_df
        .filter(
            (pl.col("date") >= pd.to_datetime(window_start)) &
            (pl.col("date") <= pd.to_datetime(window_end)) &
            pl.col("price_usd").is_not_null()
        )
        .sort("date")
    )

    if window_btc_df.is_empty():
        raise ValueError(f"No BTC data available for {window_start} to {window_end}")

    strategy = stacksats_strategy_objects[strategy_key]

    export_obj = runner.export(
        strategy,
        config,
        btc_df=window_btc_df,
    )

    weights = export_obj.to_dataframe()

    if not isinstance(weights, pl.DataFrame):
        weights = pl.from_pandas(weights)

    weights = weights.with_columns([
        pl.col("start_date").cast(pl.Datetime),
        pl.col("end_date").cast(pl.Datetime),
        pl.col("date").cast(pl.Datetime),
    ])

    latest_end = weights.select(pl.col("end_date").max()).item()

    one_window = (
        weights
        .filter(pl.col("end_date") == latest_end)
        .sort("date")
        .select(["date", "weight"])
        .rename({"weight": "raw_weight"})
        .to_pandas()
    )

    one_window["date"] = pd.to_datetime(one_window["date"])

    merged = (
        window_df[["date"]]
        .merge(one_window, on="date", how="left")
        .sort_values("date")
        .reset_index(drop=True)
    )

    if merged["raw_weight"].isna().any():
        missing_dates = (
            merged.loc[merged["raw_weight"].isna(), "date"]
            .dt.strftime("%Y-%m-%d")
            .head(10)
            .tolist()
        )

        raise ValueError(
            f"Missing StackSats weights for {strategy_key} "
            f"from {window_start} to {window_end}. "
            f"Example missing dates: {missing_dates}"
        )

    # This function returns the original StackSats normalized weights.
    # The final regime-selected weights are capped later in Cell 12.
    final_weights = build_simple_normalized_weights(
        merged["raw_weight"].values
    )

    _export_cache[cache_key] = final_weights.copy()

    return final_weights


# ============================================================


## Cell 5: Feature engineering

Create rolling price features, SMA ratios, returns, drawdowns, merge processed BRK metrics, and compute causal rolling z-scores for composite signals.

In [5]:
# Cell 5: Feature engineering
# ============================================================
# This cell creates rolling features used for regime classification.
# It creates SMA values, returns, rolling highs, SMA ratios, and drawdown features.

feature_exprs = []

for d in LOOKBACK_DAYS:
    feature_exprs.extend([
        pl.col("price_usd")
        .rolling_mean(window_size=d, min_samples=max(3, int(d * 0.30)))
        .alias(f"price_{d}d_sma"),

        pl.col("price_usd")
        .pct_change(d)
        .alias(f"btc_return_{d}d"),

        pl.col("price_usd")
        .rolling_max(window_size=d, min_samples=max(3, int(d * 0.30)))
        .alias(f"rolling_high_{d}d"),
    ])

btc_df = btc_df.with_columns(feature_exprs)

ratio_exprs = []

for d in LOOKBACK_DAYS:
    ratio_exprs.extend([
        (pl.col("price_usd") / pl.col(f"price_{d}d_sma"))
        .alias(f"price_{d}d_sma_ratio"),

        ((pl.col("price_usd") / pl.col(f"rolling_high_{d}d")) - 1)
        .alias(f"drawdown_{d}d"),
    ])

btc_df = btc_df.with_columns(ratio_exprs)

btc_data = btc_df.to_pandas()
btc_data["date"] = pd.to_datetime(btc_data["date"])

feature_cols = [
    "price_usd",
    "mvrv",
    "adjusted_sopr",
    "adjusted_sopr_7d_ema",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

for d in LOOKBACK_DAYS:
    feature_cols.extend([
        f"price_{d}d_sma",
        f"price_{d}d_sma_ratio",
        f"btc_return_{d}d",
        f"drawdown_{d}d",
    ])

btc_data = (
    btc_data
    .dropna(subset=feature_cols)
    .sort_values("date")
    .reset_index(drop=True)
)

print("Cleaned date range:")
print(btc_data["date"].min(), "to", btc_data["date"].max())
print("Cleaned rows:", len(btc_data))

# ============================================================
# Composite metric merge from processed BRK metrics
# ============================================================
# bitcoin_analytics.parquet is used as the base dataframe for:
# price, StackSats Momentum, SMA, and regime features.
#
# processed_brk_metrics_path is used to add the extra composite metrics.
# Expected long format:
# day_utc | metric | value
#
# This block must run before regime classification and train/test split,
# so train_eval_df and test_eval_df contain all z_* composite columns.

if not processed_brk_metrics_path.exists():
    raise FileNotFoundError(
        f"Processed BRK metric file not found at: {processed_brk_metrics_path}. "
        "Please update processed_brk_metrics_path."
    )

processed_schema = pl.read_parquet_schema(processed_brk_metrics_path)
required_processed_cols = ["day_utc", "metric", "value"]

missing_processed_cols = [
    col for col in required_processed_cols
    if col not in processed_schema
]

if missing_processed_cols:
    raise ValueError(
        "processed_brk_metrics_path must contain day_utc, metric, and value. "
        f"Missing columns: {missing_processed_cols}"
    )

available_processed_metrics_df = (
    pl.scan_parquet(processed_brk_metrics_path)
    .select("metric")
    .unique()
    .collect()
)

available_processed_metrics = set(
    available_processed_metrics_df
    .get_column("metric")
    .drop_nulls()
    .cast(pl.Utf8)
    .to_list()
)

missing_composite_metrics = [
    metric for metric in COMPOSITE_SIGNAL_COLS
    if metric not in available_processed_metrics
]

print("Composite metric availability in processed BRK file:")
for metric in COMPOSITE_SIGNAL_COLS:
    print(f"{metric in available_processed_metrics}: {metric}")

if missing_composite_metrics:
    raise ValueError(
        "Missing composite metrics from processed BRK file: "
        f"{missing_composite_metrics}"
    )

composite_wide_df = (
    pl.scan_parquet(processed_brk_metrics_path)
    .filter(pl.col("metric").is_in(COMPOSITE_SIGNAL_COLS))
    .select("day_utc", "metric", "value")
    .collect()
    .with_columns([
        pl.col("day_utc").cast(pl.Datetime).alias("date"),
        pl.col("value").cast(pl.Float64),
    ])
    .pivot(
        values="value",
        index="date",
        columns="metric",
        aggregate_function="last",
    )
    .sort("date")
    .to_pandas()
)

composite_wide_df["date"] = pd.to_datetime(composite_wide_df["date"])

# Drop duplicate composite columns from btc_data before merging.
# The composite strategy should use the processed BRK metric version.
duplicate_composite_cols = [
    col for col in COMPOSITE_SIGNAL_COLS
    if col in btc_data.columns
]

if duplicate_composite_cols:
    btc_data = btc_data.drop(columns=duplicate_composite_cols)

btc_data = (
    btc_data
    .merge(composite_wide_df[["date"] + COMPOSITE_SIGNAL_COLS], on="date", how="left")
    .sort_values("date")
    .reset_index(drop=True)
)

print("Merged composite metrics from:", processed_brk_metrics_path)
print("Composite missing values after merge:")
print(btc_data[COMPOSITE_SIGNAL_COLS].isna().sum().to_string())

# Compute causal rolling z-scores for composite signals.
for col in COMPOSITE_SIGNAL_COLS:
    btc_data[f"z_{col}"] = rolling_zscore(
        btc_data[col],
        window=COMPOSITE_ROLLING_WINDOW,
    )

# Forward-fill only after causal rolling z-score calculation to handle sparse metrics.
# No backward fill is used.
btc_data[COMPOSITE_Z_COLS] = btc_data[COMPOSITE_Z_COLS].ffill()

# Keep only rows where composite z-scores are available.
btc_data = (
    btc_data
    .dropna(subset=COMPOSITE_Z_COLS)
    .sort_values("date")
    .reset_index(drop=True)
)

print("Date range after composite z-score preparation:")
print(btc_data["date"].min(), "to", btc_data["date"].max())
print("Rows after composite z-score preparation:", len(btc_data))

print("Composite z-score columns present in btc_data:")
print({col: col in btc_data.columns for col in COMPOSITE_Z_COLS})

composite_signal_weight_df = pd.DataFrame({
    "composite_signal": COMPOSITE_SIGNAL_COLS,
    "initial_equal_weight": [COMPOSITE_SIGNAL_WEIGHTS[col] for col in COMPOSITE_SIGNAL_COLS],
})

display(composite_signal_weight_df)


# ============================================================


Cleaned date range:
2011-08-16 00:00:00 to 2026-03-13 00:00:00
Cleaned rows: 5324
Composite metric availability in processed BRK file:
True: mvrv
True: sopr_7d_ema
True: reserve_risk
True: greed_index
True: puell_multiple
True: lth_nupl
True: sell_side_risk_ratio_7d_ema
True: net_unrealized_pnl_rel_to_market_cap


C:\Users\ragha\AppData\Local\Temp\ipykernel_30716\4258290425.py:141: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(


Merged composite metrics from: ..\data\processed\brk_metrics.parquet
Composite missing values after merge:
mvrv                                    0
sopr_7d_ema                             0
reserve_risk                            0
greed_index                             0
puell_multiple                          0
lth_nupl                                0
sell_side_risk_ratio_7d_ema             0
net_unrealized_pnl_rel_to_market_cap    0
Date range after composite z-score preparation:
2011-12-13 00:00:00 to 2026-03-13 00:00:00
Rows after composite z-score preparation: 5205
Composite z-score columns present in btc_data:
{'z_mvrv': True, 'z_sopr_7d_ema': True, 'z_reserve_risk': True, 'z_greed_index': True, 'z_puell_multiple': True, 'z_lth_nupl': True, 'z_sell_side_risk_ratio_7d_ema': True, 'z_net_unrealized_pnl_rel_to_market_cap': True}


,composite_signal,initial_equal_weight
0,mvrv,0.125
1,sopr_7d_ema,0.125
2,reserve_risk,0.125
3,greed_index,0.125
4,puell_multiple,0.125
5,lth_nupl,0.125
6,sell_side_risk_ratio_7d_ema,0.125
7,net_unrealized_pnl_rel_to_market_cap,0.125


## Cell 6: Regime classification

Assign each day to a combined market regime using BTC trend, MVRV valuation, and market-cap versus realized-cap growth.

In [6]:
# Cell 6: Regime classification
# ============================================================
# This cell labels every day with a simpler combined market regime.
# The regime combines:
# 1. BTC trend behavior
# 2. MVRV valuation behavior
# 3. realized-cap vs market-cap growth behavior

def classify_btc_mvrv_market_cap_regime(row):
    """
    Classify each day into a simpler BTC/on-chain regime.

    The regime combines:
    1. BTC trend regime using SMA ratio and returns
    2. MVRV valuation regime
    3. market-cap / realized-cap growth regime

    Returns
    -------
    str
        Combined regime label in the format:
        BTC trend regime | MVRV valuation regime | cap-growth regime
    """

    # ------------------------------------------------------------
    # BTC trend features
    # ------------------------------------------------------------
    sma_selected_ratio = row[f"price_{SMA_LOOKBACK}d_sma_ratio"]
    sma_regime_ratio = row[f"price_{REGIME_LOOKBACK}d_sma_ratio"]
    return_momentum = row[f"btc_return_{MOMENTUM_LOOKBACK}d"]
    return_sma = row[f"btc_return_{SMA_LOOKBACK}d"]

    # ------------------------------------------------------------
    # On-chain / market features
    # ------------------------------------------------------------
    mvrv = row["mvrv"]
    realized_growth = row["realized_cap_growth_rate"]
    market_growth = row["market_cap_growth_rate"]

    # ------------------------------------------------------------
    # 1. BTC trend regime without drawdown
    # ------------------------------------------------------------
    if sma_regime_ratio >= 1.05 and return_sma > 0:
        btc_regime = "BTC Bull"

    elif sma_selected_ratio <= 0.95 and return_sma < 0:
        btc_regime = "BTC Bear"

    elif sma_selected_ratio < 1.0 and return_momentum > 0:
        btc_regime = "BTC Recovery"

    else:
        btc_regime = "BTC Neutral"

    # ------------------------------------------------------------
    # 2. MVRV valuation regime
    # ------------------------------------------------------------
    if mvrv < 1.0:
        valuation_regime = "Low MVRV"

    elif mvrv > 2.5:
        valuation_regime = "High MVRV"

    else:
        valuation_regime = "Normal MVRV"

    # ------------------------------------------------------------
    # 3. Market-cap / realized-cap growth regime
    # ------------------------------------------------------------
    if realized_growth > market_growth:
        cap_regime = "Realized Growth Leading"

    else:
        cap_regime = "Market Growth Leading"

    # ------------------------------------------------------------
    # Final combined regime
    # ------------------------------------------------------------
    return btc_regime + " | " + valuation_regime + " | " + cap_regime


btc_data["combined_regime"] = btc_data.apply(
    classify_btc_mvrv_market_cap_regime,
    axis=1,
)

display(
    btc_data[["date", "price_usd", "combined_regime"]].head()
)


# ============================================================


,date,price_usd,combined_regime
0,2011-12-13,3.21,BTC Bear | Low MVRV | Realized Growth Leading
1,2011-12-14,3.12,BTC Bear | Low MVRV | Realized Growth Leading
2,2011-12-15,3.17,BTC Bear | Low MVRV | Realized Growth Leading
3,2011-12-16,3.18,BTC Bear | Low MVRV | Realized Growth Leading
4,2011-12-17,3.19,BTC Bear | Low MVRV | Realized Growth Leading


# Cell 5: Feature engineering
# ============================================================
# This cell creates rolling features used for regime classification.
# It creates SMA values, returns, rolling highs, SMA ratios, and drawdown features.

feature_exprs = []

for d in LOOKBACK_DAYS:
    feature_exprs.extend([
        pl.col("price_usd")
        .rolling_mean(window_size=d, min_samples=max(3, int(d * 0.30)))
        .alias(f"price_{d}d_sma"),

        pl.col("price_usd")
        .pct_change(d)
        .alias(f"btc_return_{d}d"),

        pl.col("price_usd")
        .rolling_max(window_size=d, min_samples=max(3, int(d * 0.30)))
        .alias(f"rolling_high_{d}d"),
    ])

btc_df = btc_df.with_columns(feature_exprs)

ratio_exprs = []

for d in LOOKBACK_DAYS:
    ratio_exprs.extend([
        (pl.col("price_usd") / pl.col(f"price_{d}d_sma"))
        .alias(f"price_{d}d_sma_ratio"),

        ((pl.col("price_usd") / pl.col(f"rolling_high_{d}d")) - 1)
        .alias(f"drawdown_{d}d"),
    ])

btc_df = btc_df.with_columns(ratio_exprs)

btc_data = btc_df.to_pandas()
btc_data["date"] = pd.to_datetime(btc_data["date"])

feature_cols = [
    "price_usd",
    "mvrv",
    "adjusted_sopr",
    "adjusted_sopr_7d_ema",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

for d in LOOKBACK_DAYS:
    feature_cols.extend([
        f"price_{d}d_sma",
        f"price_{d}d_sma_ratio",
        f"btc_return_{d}d",
        f"drawdown_{d}d",
    ])

btc_data = (
    btc_data
    .dropna(subset=feature_cols)
    .sort_values("date")
    .reset_index(drop=True)
)

print("Cleaned date range:")
print(btc_data["date"].min(), "to", btc_data["date"].max())
print("Cleaned rows:", len(btc_data))

# ============================================================
# Composite metric merge from processed BRK metrics
# ============================================================
# bitcoin_analytics.parquet is used as the base dataframe for:
# price, StackSats Momentum, SMA, and regime features.
#
# processed_brk_metrics_path is used to add the extra composite metrics.
# Expected long format:
# day_utc | metric | value
#
# This block must run before regime classification and train/test split,
# so train_eval_df and test_eval_df contain all z_* composite columns.

if not processed_brk_metrics_path.exists():
    raise FileNotFoundError(
        f"Processed BRK metric file not found at: {processed_brk_metrics_path}. "
        "Please update processed_brk_metrics_path."
    )

processed_schema = pl.read_parquet_schema(processed_brk_metrics_path)
required_processed_cols = ["day_utc", "metric", "value"]

missing_processed_cols = [
    col for col in required_processed_cols
    if col not in processed_schema
]

if missing_processed_cols:
    raise ValueError(
        "processed_brk_metrics_path must contain day_utc, metric, and value. "
        f"Missing columns: {missing_processed_cols}"
    )

available_processed_metrics_df = (
    pl.scan_parquet(processed_brk_metrics_path)
    .select("metric")
    .unique()
    .collect()
)

available_processed_metrics = set(
    available_processed_metrics_df
    .get_column("metric")
    .drop_nulls()
    .cast(pl.Utf8)
    .to_list()
)

missing_composite_metrics = [
    metric for metric in COMPOSITE_SIGNAL_COLS
    if metric not in available_processed_metrics
]

print("Composite metric availability in processed BRK file:")
for metric in COMPOSITE_SIGNAL_COLS:
    print(f"{metric in available_processed_metrics}: {metric}")

if missing_composite_metrics:
    raise ValueError(
        "Missing composite metrics from processed BRK file: "
        f"{missing_composite_metrics}"
    )

composite_wide_df = (
    pl.scan_parquet(processed_brk_metrics_path)
    .filter(pl.col("metric").is_in(COMPOSITE_SIGNAL_COLS))
    .select("day_utc", "metric", "value")
    .collect()
    .with_columns([
        pl.col("day_utc").cast(pl.Datetime).alias("date"),
        pl.col("value").cast(pl.Float64),
    ])
    .pivot(
        values="value",
        index="date",
        columns="metric",
        aggregate_function="last",
    )
    .sort("date")
    .to_pandas()
)

composite_wide_df["date"] = pd.to_datetime(composite_wide_df["date"])

# Drop duplicate composite columns from btc_data before merging.
# The composite strategy should use the processed BRK metric version.
duplicate_composite_cols = [
    col for col in COMPOSITE_SIGNAL_COLS
    if col in btc_data.columns
]

if duplicate_composite_cols:
    btc_data = btc_data.drop(columns=duplicate_composite_cols)

btc_data = (
    btc_data
    .merge(composite_wide_df[["date"] + COMPOSITE_SIGNAL_COLS], on="date", how="left")
    .sort_values("date")
    .reset_index(drop=True)
)

print("Merged composite metrics from:", processed_brk_metrics_path)
print("Composite missing values after merge:")
print(btc_data[COMPOSITE_SIGNAL_COLS].isna().sum().to_string())

# Compute causal rolling z-scores for composite signals.
for col in COMPOSITE_SIGNAL_COLS:
    btc_data[f"z_{col}"] = rolling_zscore(
        btc_data[col],
        window=COMPOSITE_ROLLING_WINDOW,
    )

# Forward-fill only after causal rolling z-score calculation to handle sparse metrics.
# No backward fill is used.
btc_data[COMPOSITE_Z_COLS] = btc_data[COMPOSITE_Z_COLS].ffill()

# Keep only rows where composite z-scores are available.
btc_data = (
    btc_data
    .dropna(subset=COMPOSITE_Z_COLS)
    .sort_values("date")
    .reset_index(drop=True)
)

print("Date range after composite z-score preparation:")
print(btc_data["date"].min(), "to", btc_data["date"].max())
print("Rows after composite z-score preparation:", len(btc_data))

print("Composite z-score columns present in btc_data:")
print({col: col in btc_data.columns for col in COMPOSITE_Z_COLS})

composite_signal_weight_df = pd.DataFrame({
    "composite_signal": COMPOSITE_SIGNAL_COLS,
    "initial_equal_weight": [COMPOSITE_SIGNAL_WEIGHTS[col] for col in COMPOSITE_SIGNAL_COLS],
})

display(composite_signal_weight_df)


# ============================================================


## Cell 7: Train/test split

Split the cleaned data into training and testing periods, then trim both into complete 365-day windows so each window receives the same budget.

In [7]:
# Cell 7: Train/test split
# ============================================================
# This cell splits data into train and test periods.
# It also trims each split into complete 365-day evaluation windows.

raw_train_df = btc_data[
    (btc_data["date"] >= pd.to_datetime(TRAIN_START)) &
    (btc_data["date"] <= pd.to_datetime(TRAIN_END))
].copy().reset_index(drop=True)

raw_test_df = btc_data[
    (btc_data["date"] >= pd.to_datetime(TEST_START)) &
    (btc_data["date"] <= pd.to_datetime(TEST_END))
].copy().reset_index(drop=True)

train_eval_df, n_train_windows, train_remainder = trim_full_windows(raw_train_df, WINDOW_SIZE)
test_eval_df, n_test_windows, test_remainder = trim_full_windows(raw_test_df, WINDOW_SIZE)

split_summary_df = pd.DataFrame([
    {
        "split": "train",
        "start_date": raw_train_df["date"].min(),
        "end_date": raw_train_df["date"].max(),
        "rows_total": len(raw_train_df),
        "rows_eval": len(train_eval_df),
        "windows": n_train_windows,
        "remainder_dropped": train_remainder,
        "budget_rule": "$1,000 per 365-day training window",
    },
    {
        "split": "test",
        "start_date": raw_test_df["date"].min(),
        "end_date": raw_test_df["date"].max(),
        "rows_total": len(raw_test_df),
        "rows_eval": len(test_eval_df),
        "windows": n_test_windows,
        "remainder_dropped": test_remainder,
        "budget_rule": "$1,000 per 365-day test window",
    },
])

display(split_summary_df)

print("Composite z-score columns present in train_eval_df:")
print({col: col in train_eval_df.columns for col in COMPOSITE_Z_COLS})

print("Composite z-score columns present in test_eval_df:")
print({col: col in test_eval_df.columns for col in COMPOSITE_Z_COLS})


# ============================================================


,split,start_date,end_date,rows_total,rows_eval,windows,remainder_dropped,budget_rule
0,train,2018-01-01,2023-12-31,2191,2190,6,1,"$1,000 per 365-day training window"
1,test,2024-01-01,2025-12-31,731,730,2,1,"$1,000 per 365-day test window"


Composite z-score columns present in train_eval_df:
{'z_mvrv': True, 'z_sopr_7d_ema': True, 'z_reserve_risk': True, 'z_greed_index': True, 'z_puell_multiple': True, 'z_lth_nupl': True, 'z_sell_side_risk_ratio_7d_ema': True, 'z_net_unrealized_pnl_rel_to_market_cap': True}
Composite z-score columns present in test_eval_df:
{'z_mvrv': True, 'z_sopr_7d_ema': True, 'z_reserve_risk': True, 'z_greed_index': True, 'z_puell_multiple': True, 'z_lth_nupl': True, 'z_sell_side_risk_ratio_7d_ema': True, 'z_net_unrealized_pnl_rel_to_market_cap': True}


## Cell 7AA: Optimize composite strategy with Nelder-Mead

Optimize composite signal weights and gamma using the training period only. The test period is not used during optimization.

In [8]:
# Cell 7AA: Optimize composite strategy with Nelder-Mead
# ============================================================
# This cell optimizes the composite strategy using the training period only.
#
# It follows the same idea as the composite signal strategy:
# 1. Start from equal signal weights.
# 2. Use causal rolling z-scores.
# 3. Optimize signal weights and gamma with Nelder-Mead.
# 4. Maximize train-period sats-per-dollar improvement versus DCA.
#
# No test-period performance is used during optimization.

def softmax_np(x):
    """
    Convert unconstrained values into positive weights that sum to 1.
    """
    x = np.asarray(x, dtype=float)
    x = x - np.max(x)
    exp_x = np.exp(x)
    return exp_x / exp_x.sum()


def composite_weights_from_params(window_df, params):
    """
    Build composite allocation weights for one window from optimizer parameters.

    Parameters
    ----------
    window_df : pd.DataFrame
        One 365-day window with composite z-score columns.
    params : np.ndarray
        Nelder-Mead parameter vector:
        - first N values are signal-weight logits
        - final value controls gamma

    Returns
    -------
    tuple[np.ndarray, np.ndarray, float]
        Daily composite weights, optimized signal weights, and gamma.
    """
    n_signals = len(COMPOSITE_SIGNAL_COLS)

    signal_logits = params[:n_signals]
    raw_gamma = params[n_signals]

    signal_weights = softmax_np(signal_logits)

    gamma = float(
        np.clip(
            np.exp(raw_gamma),
            COMPOSITE_GAMMA_MIN,
            COMPOSITE_GAMMA_MAX,
        )
    )

    z_matrix = (
        window_df[COMPOSITE_Z_COLS]
        .fillna(0.0)
        .to_numpy(dtype=float)
    )

    # Lower z-score means cheaper/stressed market condition.
    # Negative z-scores therefore increase cheapness.
    cheapness = z_matrix @ (-signal_weights)

    buy_score = np.maximum(0.0, cheapness) ** gamma

    multiplier = np.maximum(
        SIGNAL_FLOOR,
        1.0 + COMPOSITE_SIGNAL_STRENGTH * buy_score,
    )

    weights = build_simple_normalized_weights(multiplier)

    return weights, signal_weights, gamma


def evaluate_composite_params_on_train(params, train_df):
    """
    Evaluate composite strategy parameters on all train windows.
    """
    n_windows = len(train_df) // WINDOW_SIZE

    if n_windows == 0:
        raise ValueError("No full training windows available for composite optimization.")

    strategy_spd_values = []
    dca_spd_values = []

    for window_idx in range(n_windows):
        start = window_idx * WINDOW_SIZE
        end = start + WINDOW_SIZE

        window_df = train_df.iloc[start:end].copy().reset_index(drop=True)

        composite_weights, _, _ = composite_weights_from_params(window_df, params)
        dca_weights = np.full(len(window_df), 1.0 / len(window_df))

        strategy_sats = (
            composite_weights
            * TOTAL_BUDGET_USD
            / window_df["price_usd"].to_numpy(dtype=float)
            * 100_000_000
        ).sum()

        dca_sats = (
            dca_weights
            * TOTAL_BUDGET_USD
            / window_df["price_usd"].to_numpy(dtype=float)
            * 100_000_000
        ).sum()

        strategy_spd_values.append(strategy_sats / TOTAL_BUDGET_USD)
        dca_spd_values.append(dca_sats / TOTAL_BUDGET_USD)

    strategy_spd_sum = float(np.sum(strategy_spd_values))
    dca_spd_sum = float(np.sum(dca_spd_values))

    improvement_pct = (strategy_spd_sum / dca_spd_sum - 1.0) * 100.0

    return {
        "strategy_spd_sum": strategy_spd_sum,
        "dca_spd_sum": dca_spd_sum,
        "improvement_pct": improvement_pct,
    }


def composite_objective(params):
    """
    Nelder-Mead objective.

    Returns
    -------
    float
        Negative train improvement. Minimize this to maximize train improvement.
    """
    metrics = evaluate_composite_params_on_train(params, train_eval_df)
    return -metrics["improvement_pct"]


initial_composite_weight_vector = np.asarray(
    [COMPOSITE_SIGNAL_WEIGHTS[col] for col in COMPOSITE_SIGNAL_COLS],
    dtype=float,
)

initial_composite_weight_vector = (
    initial_composite_weight_vector
    / initial_composite_weight_vector.sum()
)

initial_composite_logits = np.log(initial_composite_weight_vector + 1e-12)

initial_composite_gamma_param = np.log(
    np.clip(
        COMPOSITE_GAMMA,
        COMPOSITE_GAMMA_MIN,
        COMPOSITE_GAMMA_MAX,
    )
)

initial_composite_params = np.concatenate([
    initial_composite_logits,
    np.asarray([initial_composite_gamma_param]),
])

initial_composite_metrics = evaluate_composite_params_on_train(
    initial_composite_params,
    train_eval_df,
)

print("Initial composite train improvement %:", round(initial_composite_metrics["improvement_pct"], 6))
print("Initial composite gamma:", COMPOSITE_GAMMA)
print("Initial equal signal weights:")
display(
    pd.DataFrame({
        "signal": COMPOSITE_SIGNAL_COLS,
        "initial_weight": initial_composite_weight_vector,
    }).round(6)
)

composite_optimization_result = minimize(
    composite_objective,
    initial_composite_params,
    method="Nelder-Mead",
    options={
        "maxiter": COMPOSITE_OPT_MAXITER,
        "maxfev": COMPOSITE_OPT_MAXFEV,
        "xatol": 1e-5,
        "fatol": 1e-5,
        "disp": True,
    },
)

optimized_composite_params = composite_optimization_result.x

optimized_composite_signal_weights = softmax_np(
    optimized_composite_params[:len(COMPOSITE_SIGNAL_COLS)]
)

optimized_composite_gamma = float(
    np.clip(
        np.exp(optimized_composite_params[len(COMPOSITE_SIGNAL_COLS)]),
        COMPOSITE_GAMMA_MIN,
        COMPOSITE_GAMMA_MAX,
    )
)

optimized_composite_metrics = evaluate_composite_params_on_train(
    optimized_composite_params,
    train_eval_df,
)

COMPOSITE_SIGNAL_WEIGHTS = {
    signal: float(weight)
    for signal, weight in zip(COMPOSITE_SIGNAL_COLS, optimized_composite_signal_weights)
}

COMPOSITE_GAMMA = optimized_composite_gamma

composite_optimization_summary_df = pd.DataFrame([
    {
        "stage": "initial",
        "train_improvement_pct": initial_composite_metrics["improvement_pct"],
        "gamma": float(np.exp(initial_composite_gamma_param)),
        "success": None,
        "message": "starting parameters",
    },
    {
        "stage": "optimized",
        "train_improvement_pct": optimized_composite_metrics["improvement_pct"],
        "gamma": COMPOSITE_GAMMA,
        "success": composite_optimization_result.success,
        "message": composite_optimization_result.message,
    },
])

optimized_composite_weights_df = pd.DataFrame({
    "signal": COMPOSITE_SIGNAL_COLS,
    "initial_weight": initial_composite_weight_vector,
    "optimized_weight": optimized_composite_signal_weights,
})

print("Nelder-Mead optimization complete for composite strategy.")
display(composite_optimization_summary_df.round(6))
display(optimized_composite_weights_df.round(6))


# ============================================================


Initial composite train improvement %: 8.849454
Initial composite gamma: 1.0
Initial equal signal weights:


,signal,initial_weight
0,mvrv,0.125
1,sopr_7d_ema,0.125
2,reserve_risk,0.125
3,greed_index,0.125
4,puell_multiple,0.125
5,lth_nupl,0.125
6,sell_side_risk_ratio_7d_ema,0.125
7,net_unrealized_pnl_rel_to_market_cap,0.125


Optimization terminated successfully.
         Current function value: -23.384509
         Iterations: 263
         Function evaluations: 615
Nelder-Mead optimization complete for composite strategy.


,stage,train_improvement_pct,gamma,success,message
0,initial,8.849454,1.0,None,starting parameters
1,optimized,23.384509,3.0,True,Optimization terminated successfully.


,signal,initial_weight,optimized_weight
0,mvrv,0.125,0.0
1,sopr_7d_ema,0.125,0.0
2,reserve_risk,0.125,0.0
3,greed_index,0.125,0.0
4,puell_multiple,0.125,0.0
5,lth_nupl,0.125,1.0
6,sell_side_risk_ratio_7d_ema,0.125,0.0
7,net_unrealized_pnl_rel_to_market_cap,0.125,0.0


## Cell 8: Candidate strategy weights

Create daily weights for each candidate strategy: StackSats MVRV, StackSats Momentum, SMA, and optimized Composite.

In [9]:
# Cell 8: Candidate strategy weights
# ============================================================
# This cell creates daily weights for each candidate strategy.
#
# Candidate strategies:
# 1. StackSats MVRV
# 2. StackSats Momentum
# 3. SMA
# 4. Composite on-chain strategy

def add_composite_signal_to_dataframe(df):
    """
    Create composite on-chain strategy weights using optimized Nelder-Mead parameters.

    Composite logic:
    1. Use causal rolling z-scores for each processed BRK signal.
    2. Apply optimized signal weights learned from training windows only.
    3. Negate z-scores so lower/stressed values become stronger cheapness.
    4. Apply optimized gamma.
    5. Convert cheapness into a positive multiplier.
    6. Normalize into daily allocation weights.
    """
    out = df.copy()

    weight_vector = np.asarray(
        [COMPOSITE_SIGNAL_WEIGHTS[col] for col in COMPOSITE_SIGNAL_COLS],
        dtype=float,
    )
    weight_vector = weight_vector / weight_vector.sum()

    z_matrix = (
        out[COMPOSITE_Z_COLS]
        .fillna(0.0)
        .to_numpy(dtype=float)
    )

    out["composite_cheapness"] = z_matrix @ (-weight_vector)

    out["composite_buy_score"] = (
        np.maximum(0.0, out["composite_cheapness"].values)
        ** COMPOSITE_GAMMA
    )

    out["composite_multiplier"] = np.maximum(
        SIGNAL_FLOOR,
        1.0 + COMPOSITE_SIGNAL_STRENGTH * out["composite_buy_score"].values,
    )

    out["composite_weight"] = build_simple_normalized_weights(
        out["composite_multiplier"].values
    )

    return out


def create_candidate_strategy_weights_simple(data):
    """
    Create daily allocation weights for all candidate strategies.

    Parameters
    ----------
    data : pd.DataFrame
        One 365-day window of BTC data with regime and engineered features.

    Returns
    -------
    pd.DataFrame
        Original data plus strategy weight columns:
        - dca_weight
        - stacksats_mvrv_weight
        - stacksats_momentum_weight
        - sma_{SMA_LOOKBACK}d_weight
        - composite_weight
    """
    df = data.copy().sort_values("date").reset_index(drop=True)
    n = len(df)

    if n == 0:
        raise ValueError("No data available.")

    # Defensive check for composite z-score columns.
    # If this fails, re-run the notebook from the top because the composite merge
    # and z-score block must run before regime classification and train/test split.
    missing_composite_z_cols = [
        col for col in COMPOSITE_Z_COLS
        if col not in df.columns
    ]

    if missing_composite_z_cols:
        raise KeyError(
            "Missing composite z-score columns in this strategy window: "
            f"{missing_composite_z_cols}. "
            "Re-run the notebook from the top so the composite merge/z-score block "
            "executes before train_eval_df and test_eval_df are created."
        )

    # Uniform DCA benchmark weight.
    df["dca_weight"] = 1.0 / n


    # Candidate 1: StackSats MVRV strategy.
    df["stacksats_mvrv_weight"] = export_stacksats_weights_for_window(
        strategy_key="stacksats_mvrv_weight",
        window_df=df,
        full_btc_df=btc_df,
    )

    # Candidate 2: StackSats Momentum strategy.
    df["stacksats_momentum_weight"] = export_stacksats_weights_for_window(
        strategy_key="stacksats_momentum_weight",
        window_df=df,
        full_btc_df=btc_df,
    )

    # Candidate 3: Custom SMA strategy.
    sma_signal = (1.0 - df[f"price_{SMA_LOOKBACK}d_sma_ratio"]).clip(-1, 1)
    sma_multiplier = np.maximum(SIGNAL_FLOOR, 1.0 + 1.50 * sma_signal)

    df[f"sma_{SMA_LOOKBACK}d_weight"] = build_simple_normalized_weights(
        sma_multiplier.values
    )

    # Candidate 4: Composite on-chain strategy.

    df = add_composite_signal_to_dataframe(df)

    return df


# ============================================================


## Cell 9: Evaluate strategies by regime

Evaluate each candidate strategy within each regime across training windows and compare against uniform DCA using sats and sats per dollar.

In [10]:
# Cell 9: Evaluate strategies by regime
# ============================================================
# This cell evaluates each candidate strategy within each regime across training windows.
# It compares each candidate against DCA using sats accumulated and sats per dollar.

def evaluate_strategies_by_regime_in_365_windows(
    data,
    total_budget_usd=TOTAL_BUDGET_USD,
):
    """
    Evaluate all candidate strategies inside each regime for every training window.

    Parameters
    ----------
    data : pd.DataFrame
        Training dataframe trimmed to complete 365-day windows.
    total_budget_usd : float
        Budget allocated per 365-day window.

    Returns
    -------
    pd.DataFrame
        Regime-level strategy performance versus DCA.
    """
    n_windows = len(data) // WINDOW_SIZE
    rows = []

    for window_idx in range(n_windows):
        start = window_idx * WINDOW_SIZE
        end = start + WINDOW_SIZE

        window_data = data.iloc[start:end].copy().reset_index(drop=True)
        df = create_candidate_strategy_weights_simple(data=window_data)

        for regime, regime_df in df.groupby("combined_regime"):

            if len(regime_df) < MIN_REGIME_DAYS:
                continue

            dca_sats = (
                regime_df["dca_weight"]
                * total_budget_usd
                / regime_df["price_usd"]
                * 100_000_000
            ).sum()

            for col in CANDIDATE_COLS:
                strategy_sats = (
                    regime_df[col]
                    * total_budget_usd
                    / regime_df["price_usd"]
                    * 100_000_000
                ).sum()

                strategy_spd = strategy_sats / total_budget_usd
                dca_spd = dca_sats / total_budget_usd

                extra_sats = strategy_sats - dca_sats
                extra_spd = strategy_spd - dca_spd

                spd_ratio = strategy_spd / dca_spd
                improvement_pct = (spd_ratio - 1.0) * 100.0

                rows.append({
                    "train_window": window_idx + 1,
                    "combined_regime": regime,
                    "days": len(regime_df),
                    "strategy": col,

                    "strategy_sats": strategy_sats,
                    "dca_sats": dca_sats,
                    "extra_sats_vs_dca": extra_sats,

                    "strategy_spd": strategy_spd,
                    "dca_spd": dca_spd,
                    "extra_spd_vs_dca": extra_spd,
                    "spd_ratio": spd_ratio,
                    "improvement_pct": improvement_pct,

                    "status": get_status_from_pct_diff(improvement_pct),
                })

    return pd.DataFrame(rows)


train_regime_results_df = evaluate_strategies_by_regime_in_365_windows(
    data=train_eval_df,
    total_budget_usd=TOTAL_BUDGET_USD,
)

display(
    train_regime_results_df
    .sort_values(["combined_regime", "improvement_pct"], ascending=[True, False])
    .round(6)
)


# ============================================================


,train_window,combined_regime,days,strategy,strategy_sats,dca_sats,extra_sats_vs_dca,strategy_spd,dca_spd,extra_spd_vs_dca,spd_ratio,improvement_pct,status
23,2,BTC Bear | Low MVRV | Realized Growth Leading,91,composite_weight,1.932015e+07,6.659642e+06,1.266051e+07,19320.154905,6659.642476,12660.512429,2.901080,190.107990,better
3,1,BTC Bear | Low MVRV | Realized Growth Leading,43,composite_weight,6.251273e+06,3.128389e+06,3.122883e+06,6251.272761,3128.389495,3122.883267,1.998240,99.823992,better
22,2,BTC Bear | Low MVRV | Realized Growth Leading,91,sma_180d_weight,1.131232e+07,6.659642e+06,4.652674e+06,11312.316775,6659.642476,4652.674299,1.698637,69.863725,better
2,1,BTC Bear | Low MVRV | Realized Growth Leading,43,sma_180d_weight,4.290631e+06,3.128389e+06,1.162242e+06,4290.631177,3128.389495,1162.241682,1.371514,37.151438,better
1,1,BTC Bear | Low MVRV | Realized Growth Leading,43,stacksats_momentum_weight,3.586709e+06,3.128389e+06,4.583191e+05,3586.708589,3128.389495,458.319094,1.146503,14.650321,better
...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,4,BTC Recovery | Normal MVRV | Market Growth Lea...,23,stacksats_mvrv_weight,5.507978e+02,1.509035e+05,-1.503527e+05,0.550798,150.903515,-150.352718,0.003650,-99.635000,worse
18,1,BTC Recovery | Normal MVRV | Realized Growth L...,33,sma_180d_weight,1.062017e+06,1.029631e+06,3.238608e+04,1062.017336,1029.631257,32.386079,1.031454,3.145406,better
16,1,BTC Recovery | Normal MVRV | Realized Growth L...,33,stacksats_mvrv_weight,9.812157e+05,1.029631e+06,-4.841555e+04,981.215706,1029.631257,-48.415551,0.952978,-4.702222,worse
17,1,BTC Recovery | Normal MVRV | Realized Growth L...,33,stacksats_momentum_weight,8.277917e+05,1.029631e+06,-2.018396e+05,827.791650,1029.631257,-201.839607,0.803969,-19.603096,worse


## Cell 10: Candidate win rate by regime

Summarize how often each candidate beats DCA across regime cases in the training period.

In [11]:
# Cell 10: Candidate win rate by regime
# ============================================================
# This cell summarizes how often each candidate strategy beats DCA across regime cases.

candidate_regime_winrate_df = (
    train_regime_results_df
    .groupby("strategy", as_index=False)
    .agg(
        regime_cases=("status", "count"),
        wins=("status", lambda x: (x == "better").sum()),
        losses=("status", lambda x: (x == "worse").sum()),
        ties=("status", lambda x: (x == "tie").sum()),
        avg_improvement_pct=("improvement_pct", "mean"),
    )
)

candidate_regime_winrate_df["win_rate_pct"] = (
    candidate_regime_winrate_df["wins"]
    / candidate_regime_winrate_df["regime_cases"]
    * 100.0
)

display(
    candidate_regime_winrate_df
    .sort_values("win_rate_pct", ascending=False)
    .round(6)
)


# ============================================================


,strategy,regime_cases,wins,losses,ties,avg_improvement_pct,win_rate_pct
1,sma_180d_weight,27,17,10,0,7.954739,62.962963
2,stacksats_momentum_weight,27,10,16,1,0.189741,37.037037
0,composite_weight,27,8,19,0,4.185528,29.629630
3,stacksats_mvrv_weight,27,8,15,4,6.494773,29.629630


## Cell 11: Learn best strategy per regime

Learn the best candidate strategy for each regime based on average training-period improvement percentage.

In [12]:
# Cell 11: Learn best strategy per regime
# ============================================================
# This cell learns the best strategy for each regime using training data.
# It selects the candidate with the highest mean improvement percentage for each regime.

best_mapping_df = (
    train_regime_results_df
    .groupby(["combined_regime", "strategy"], as_index=False)
    .agg(
        total_days=("days", "sum"),
        mean_improvement_pct=("improvement_pct", "mean"),
        median_improvement_pct=("improvement_pct", "median"),
        mean_extra_spd_vs_dca=("extra_spd_vs_dca", "mean"),
        total_extra_sats_vs_dca=("extra_sats_vs_dca", "sum"),
        windows_seen=("train_window", "nunique"),
    )
)

best_mapping_df = (
    best_mapping_df
    .sort_values(
        ["combined_regime", "mean_improvement_pct", "total_days"],
        ascending=[True, False, False],
    )
    .groupby("combined_regime")
    .head(1)
    .reset_index(drop=True)
)

best_mapping_df["status"] = best_mapping_df["mean_improvement_pct"].apply(
    get_status_from_pct_diff
)

display(
    best_mapping_df
    .sort_values("mean_improvement_pct", ascending=False)
    .round(6)
)


# ============================================================


,combined_regime,strategy,total_days,mean_improvement_pct,median_improvement_pct,mean_extra_spd_vs_dca,total_extra_sats_vs_dca,windows_seen,status
3,BTC Bull | High MVRV | Market Growth Leading,stacksats_mvrv_weight,195,162.156399,162.156399,605.516686,1.211033e+06,2,better
1,BTC Bear | Normal MVRV | Market Growth Leading,stacksats_mvrv_weight,98,108.676822,108.676822,1692.566807,3.385134e+06,2,better
0,BTC Bear | Low MVRV | Realized Growth Leading,composite_weight,300,93.140441,99.823992,5176.852762,1.553056e+07,3,better
8,BTC Recovery | Normal MVRV | Market Growth Lea...,sma_180d_weight,23,78.593313,78.593313,118.600073,1.186001e+05,1,better
6,BTC Neutral | Normal MVRV | Market Growth Leading,stacksats_mvrv_weight,229,71.386210,72.026021,545.113657,2.180455e+06,4,better
7,BTC Neutral | Normal MVRV | Realized Growth Le...,composite_weight,180,48.112671,-4.669162,-449.939466,-2.249697e+06,5,better
2,BTC Bear | Normal MVRV | Realized Growth Leading,composite_weight,333,25.895854,25.895854,1512.795598,3.025591e+06,2,better
9,BTC Recovery | Normal MVRV | Realized Growth L...,sma_180d_weight,33,3.145406,3.145406,32.386079,3.238608e+04,1,better
5,BTC Bull | Normal MVRV | Realized Growth Leading,stacksats_momentum_weight,238,1.994961,-2.257035,-23.668253,-9.467301e+04,4,better
4,BTC Bull | Normal MVRV | Market Growth Leading,stacksats_momentum_weight,409,-3.839734,-0.643755,-95.649032,-2.869471e+05,3,worse


## Cell 12: Apply trained regime mapping

Apply the learned regime-to-strategy mapping to each 365-day window and enforce the maximum daily allocation cap using MAX_DCA_MULTIPLE.

In [13]:
# Cell 12: Apply trained regime mapping
# ============================================================
# This cell defines functions to apply the learned regime-to-strategy mapping.
# For each day, it selects the strategy assigned to that day's regime.

def apply_regime_mapping_to_one_window(
    window_data,
    best_mapping_df,
    total_budget_usd=TOTAL_BUDGET_USD,
    window_number=None,
):
    """
    Apply the learned regime mapping to one 365-day window.

    Parameters
    ----------
    window_data : pd.DataFrame
        One 365-day window of BTC data.
    best_mapping_df : pd.DataFrame
        Learned mapping from regime to best strategy.
    total_budget_usd : float
        Budget allocated to the window.
    window_number : int or None
        Optional window number to add to the output.

    Returns
    -------
    pd.DataFrame
        Daily output with selected strategy, final weights, USD allocation,
        BTC accumulated, and sats accumulated.
    """
    df = create_candidate_strategy_weights_simple(data=window_data)

    regime_to_strategy = dict(
        zip(best_mapping_df["combined_regime"], best_mapping_df["strategy"])
    )

    df["selected_strategy"] = (
        df["combined_regime"]
        .map(regime_to_strategy)
        .fillna(FALLBACK_STRATEGY)
    )

    df["raw_selected_weight"] = df.apply(
        lambda row: row[row["selected_strategy"]],
        axis=1,
    )

    # ------------------------------------------------------------
    # Apply allocation cap: no final daily weight can exceed MAX_DCA_MULTIPLE x DCA.
    # ------------------------------------------------------------
    # This prevents one day from receiving an extreme allocation.
    # For a 365-day window:
    # DCA weight = 1 / 365 = 0.0027397
    # If MAX_DCA_MULTIPLE = 15, cap = 15 / 365 = 0.041096
    #
    # Important:
    # 1. First normalize the selected raw strategy weights.
    # 2. Then cap weights at MAX_DCA_MULTIPLE * DCA weight.
    # 3. Redistribute leftover budget across uncapped days.
    # 4. Repeat until all weights obey the cap and sum to 1.
    # ------------------------------------------------------------
    def normalize_with_max_dca_cap(raw_weights, dca_weights, max_dca_multiple=MAX_DCA_MULTIPLE):
        raw_weights = np.asarray(raw_weights, dtype=float)
        dca_weights = np.asarray(dca_weights, dtype=float)

        clean_weights = np.nan_to_num(
            raw_weights,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        clean_weights = np.maximum(clean_weights, 0.0)

        if clean_weights.sum() <= 0:
            clean_weights = np.ones_like(clean_weights, dtype=float)

        weights = clean_weights / clean_weights.sum()
        cap = max_dca_multiple * dca_weights

        # Iterative capped normalization.
        # Capped days are fixed at cap; the remaining budget is redistributed
        # proportionally among uncapped days.
        final_weights = weights.copy()
        capped_mask = np.zeros(len(final_weights), dtype=bool)

        for _ in range(len(final_weights) + 1):
            over_cap = final_weights > cap + 1e-12

            if not over_cap.any():
                break

            capped_mask = capped_mask | over_cap
            final_weights[capped_mask] = cap[capped_mask]

            remaining_budget = 1.0 - final_weights[capped_mask].sum()
            uncapped_mask = ~capped_mask

            if remaining_budget <= 0 or not uncapped_mask.any():
                final_weights = final_weights / final_weights.sum()
                final_weights = np.minimum(final_weights, cap)
                final_weights = final_weights / final_weights.sum()
                break

            uncapped_raw = clean_weights[uncapped_mask]

            if uncapped_raw.sum() <= 0:
                final_weights[uncapped_mask] = remaining_budget / uncapped_mask.sum()
            else:
                final_weights[uncapped_mask] = (
                    uncapped_raw / uncapped_raw.sum() * remaining_budget
                )

        # Final safety normalization.
        final_weights = np.maximum(final_weights, 0.0)
        final_weights = final_weights / final_weights.sum()

        if final_weights.max() > cap.max() + 1e-8:
            raise ValueError(
                "Allocation cap failed. Check MAX_DCA_MULTIPLE and weight logic."
            )

        return final_weights

    df["uncapped_strategy_weight"] = (
        df["raw_selected_weight"] / df["raw_selected_weight"].sum()
    )

    df["strategy_weight_cap"] = MAX_DCA_MULTIPLE * df["dca_weight"]

    df["final_strategy_weight"] = normalize_with_max_dca_cap(
        raw_weights=df["raw_selected_weight"].values,
        dca_weights=df["dca_weight"].values,
        max_dca_multiple=MAX_DCA_MULTIPLE,
    )

    df["was_capped"] = df["uncapped_strategy_weight"] > df["strategy_weight_cap"]

    df["final_strategy_usd"] = df["final_strategy_weight"] * total_budget_usd
    df["dca_usd"] = df["dca_weight"] * total_budget_usd

    df["btc_accum_strategy"] = df["final_strategy_usd"] / df["price_usd"]
    df["btc_accum_dca"] = df["dca_usd"] / df["price_usd"]

    df["sats_accum_strategy"] = df["btc_accum_strategy"] * 100_000_000
    df["sats_accum_dca"] = df["btc_accum_dca"] * 100_000_000

    df["strategy_spd_daily"] = df["sats_accum_strategy"] / total_budget_usd
    df["dca_spd_daily"] = df["sats_accum_dca"] / total_budget_usd

    if window_number is not None:
        df["window"] = window_number

    return df


def apply_regime_mapping_to_window_set(
    eval_df,
    best_mapping_df,
    total_budget_usd=TOTAL_BUDGET_USD,
):
    """
    Apply the learned regime mapping to every complete 365-day window.

    Parameters
    ----------
    eval_df : pd.DataFrame
        Train or test dataframe trimmed to complete 365-day windows.
    best_mapping_df : pd.DataFrame
        Learned regime-to-strategy mapping.
    total_budget_usd : float
        Budget allocated to each 365-day window.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        Daily output dataframe and window-level summary dataframe.
    """
    n_windows = len(eval_df) // WINDOW_SIZE
    window_dfs = []
    window_summary_rows = []

    for window_idx in range(n_windows):
        start = window_idx * WINDOW_SIZE
        end = start + WINDOW_SIZE

        window_data = eval_df.iloc[start:end].copy().reset_index(drop=True)

        window_strategy_df = apply_regime_mapping_to_one_window(
            window_data=window_data,
            best_mapping_df=best_mapping_df,
            total_budget_usd=total_budget_usd,
            window_number=window_idx + 1,
        )

        strategy_sats = window_strategy_df["sats_accum_strategy"].sum()
        dca_sats = window_strategy_df["sats_accum_dca"].sum()

        strategy_spd = strategy_sats / total_budget_usd
        dca_spd = dca_sats / total_budget_usd

        extra_sats = strategy_sats - dca_sats
        extra_spd = strategy_spd - dca_spd

        spd_ratio = strategy_spd / dca_spd
        improvement_pct = (spd_ratio - 1.0) * 100.0

        window_summary_rows.append({
            "window": window_idx + 1,
            "start_date": window_strategy_df["date"].min(),
            "end_date": window_strategy_df["date"].max(),
            "days": len(window_strategy_df),
            "budget_usd": total_budget_usd,

            "strategy_sats": strategy_sats,
            "dca_sats": dca_sats,
            "extra_sats_vs_dca": extra_sats,

            "strategy_spd": strategy_spd,
            "dca_spd": dca_spd,
            "extra_spd_vs_dca": extra_spd,
            "spd_ratio": spd_ratio,
            "improvement_pct": improvement_pct,

            "result": get_status_from_pct_diff(improvement_pct),
            "weight_sum": window_strategy_df["final_strategy_weight"].sum(),
            "max_weight": window_strategy_df["final_strategy_weight"].max(),
            "min_weight": window_strategy_df["final_strategy_weight"].min(),
            "days_above_dca_weight": int(
                (window_strategy_df["final_strategy_weight"] > window_strategy_df["dca_weight"]).sum()
            ),
        })

        window_dfs.append(window_strategy_df)

    out_df = pd.concat(window_dfs, ignore_index=True)
    summary_df = pd.DataFrame(window_summary_rows)

    return out_df, summary_df


# ============================================================


## Cell 13: Run strategy on train and test

Run the final mapped strategy on both train and test periods and compute window-level results.

In [14]:
# Cell 13: Run strategy on train and test
# ============================================================
# This cell applies the learned regime mapping to both train and test periods.

train_strategy_df, train_window_summary_df = apply_regime_mapping_to_window_set(
    train_eval_df,
    best_mapping_df,
    total_budget_usd=TOTAL_BUDGET_USD,
)

test_strategy_df, test_window_summary_df = apply_regime_mapping_to_window_set(
    test_eval_df,
    best_mapping_df,
    total_budget_usd=TOTAL_BUDGET_USD,
)

# Add StackSats-style dynamic and uniform SPD percentile columns.
# These are calculated across the 365-day windows inside each period.
train_window_summary_df = add_spd_percentiles_to_window_summary(train_window_summary_df)
test_window_summary_df = add_spd_percentiles_to_window_summary(test_window_summary_df)

display(train_window_summary_df.round(6))
display(test_window_summary_df.round(6))


# ============================================================


C:\Users\ragha\AppData\Local\Temp\ipykernel_30716\1507359995.py:22: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(train_window_summary_df.round(6))


,window,start_date,end_date,days,budget_usd,strategy_sats,dca_sats,extra_sats_vs_dca,strategy_spd,dca_spd,extra_spd_vs_dca,spd_ratio,improvement_pct,result,weight_sum,max_weight,min_weight,days_above_dca_weight,dynamic_percentile,uniform_percentile
0,1,2018-01-01,2018-12-31,365,1000.0,1.622017e+07,1.473606e+07,1.484102e+06,16220.166527,14736.064632,1484.101895,1.100712,10.071223,better,1.0,0.016112,0.000228,95,83.333333,83.333333
1,2,2019-01-01,2019-12-31,365,1000.0,1.843490e+07,1.592086e+07,2.514038e+06,18434.899450,15920.861713,2514.037737,1.157908,15.790840,better,1.0,0.026586,0.000006,88,100.000000,100.000000
2,3,2020-01-01,2020-12-30,365,1000.0,1.009913e+07,1.005557e+07,4.356272e+04,10099.133441,10055.570717,43.562724,1.004332,0.433220,better,1.0,0.041096,0.000025,24,66.666667,66.666667
3,4,2020-12-31,2021-12-30,365,1000.0,2.386055e+06,2.209056e+06,1.769988e+05,2386.055008,2209.056208,176.998800,1.080124,8.012417,better,1.0,0.041096,0.000007,68,16.666667,16.666667
4,5,2021-12-31,2022-12-30,365,1000.0,3.831625e+06,4.007365e+06,-1.757393e+05,3831.625456,4007.364728,-175.739272,0.956146,-4.385407,worse,1.0,0.019379,0.000407,104,50.000000,50.000000
5,6,2022-12-31,2023-12-30,365,1000.0,3.797242e+06,3.615539e+06,1.817034e+05,3797.242084,3615.538694,181.703390,1.050256,5.025624,better,1.0,0.012391,0.000190,65,33.333333,33.333333


C:\Users\ragha\AppData\Local\Temp\ipykernel_30716\1507359995.py:23: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(test_window_summary_df.round(6))


,window,start_date,end_date,days,budget_usd,strategy_sats,dca_sats,extra_sats_vs_dca,strategy_spd,dca_spd,extra_spd_vs_dca,spd_ratio,improvement_pct,result,weight_sum,max_weight,min_weight,days_above_dca_weight,dynamic_percentile,uniform_percentile
0,1,2024-01-01,2024-12-30,365,1000.0,1.657389e+06,1.589861e+06,67528.145274,1657.389238,1589.861093,67.528145,1.042474,4.247424,better,1.0,0.041096,0.000009,83,100.0,100.0
1,2,2024-12-31,2025-12-30,365,1000.0,1.036587e+06,9.974704e+05,39117.023924,1036.587458,997.470434,39.117024,1.039216,3.921622,better,1.0,0.025628,0.000007,59,50.0,50.0


## Cell 14: Final summary tables

Create final train/test summary tables showing total sats, SPD, improvement percentage, win rate, and percentile metrics.

In [15]:
# Cell 14: Final summary tables
# ============================================================
# This cell creates final summary tables for train and test.
# It reports total sats, SPD, improvement percentage, and window win rate.

window_winrate_df = pd.DataFrame([
    {
        "period": "train",
        "windows": len(train_window_summary_df),
        "wins": int((train_window_summary_df["result"] == "better").sum()),
        "losses": int((train_window_summary_df["result"] == "worse").sum()),
        "ties": int((train_window_summary_df["result"] == "tie").sum()),
        "win_rate_pct": (
            (train_window_summary_df["result"] == "better").sum()
            / len(train_window_summary_df)
            * 100.0
            if len(train_window_summary_df) > 0 else 0.0
        ),
    },
    {
        "period": "test",
        "windows": len(test_window_summary_df),
        "wins": int((test_window_summary_df["result"] == "better").sum()),
        "losses": int((test_window_summary_df["result"] == "worse").sum()),
        "ties": int((test_window_summary_df["result"] == "tie").sum()),
        "win_rate_pct": (
            (test_window_summary_df["result"] == "better").sum()
            / len(test_window_summary_df)
            * 100.0
            if len(test_window_summary_df) > 0 else 0.0
        ),
    },
])

display(window_winrate_df.round(2))


train_spd_summary = summarize_spd_like_composite(train_window_summary_df)
test_spd_summary = summarize_spd_like_composite(test_window_summary_df)

final_summary_df = pd.DataFrame([
    {
        "period": "train",
        "start_date": TRAIN_START,
        "end_date": TRAIN_END,
        "windows": train_spd_summary["n_windows"],
        "wins": train_spd_summary["wins"],
        "losses": train_spd_summary["losses"],
        "ties": train_spd_summary["ties"],
        "win_rate_pct": train_spd_summary["win_rate_pct"],
        "exp_decay_percentile": train_spd_summary["exp_decay_percentile"],
        "uniform_exp_decay_percentile": train_spd_summary["uniform_exp_decay_percentile"],

        "strategy_sats": train_spd_summary["strategy_sats"],
        "dca_sats": train_spd_summary["dca_sats"],
        "extra_sats_vs_dca": train_spd_summary["extra_sats_vs_dca"],

        "strategy_spd_sum": train_spd_summary["strategy_spd_sum"],
        "dca_spd_sum": train_spd_summary["dca_spd_sum"],
        "extra_spd_sum_vs_dca": train_spd_summary["extra_spd_sum_vs_dca"],

        "strategy_spd_avg": train_spd_summary["strategy_spd_avg"],
        "dca_spd_avg": train_spd_summary["dca_spd_avg"],
        "extra_spd_avg_vs_dca": train_spd_summary["extra_spd_avg_vs_dca"],

        "spd_ratio": train_spd_summary["spd_ratio"],
        "improvement_pct": train_spd_summary["improvement_pct"],
        "result": get_status_from_pct_diff(train_spd_summary["improvement_pct"]),
    },
    {
        "period": "test",
        "start_date": TEST_START,
        "end_date": TEST_END,
        "windows": test_spd_summary["n_windows"],
        "wins": test_spd_summary["wins"],
        "losses": test_spd_summary["losses"],
        "ties": test_spd_summary["ties"],
        "win_rate_pct": test_spd_summary["win_rate_pct"],
        "exp_decay_percentile": test_spd_summary["exp_decay_percentile"],
        "uniform_exp_decay_percentile": test_spd_summary["uniform_exp_decay_percentile"],

        "strategy_sats": test_spd_summary["strategy_sats"],
        "dca_sats": test_spd_summary["dca_sats"],
        "extra_sats_vs_dca": test_spd_summary["extra_sats_vs_dca"],

        "strategy_spd_sum": test_spd_summary["strategy_spd_sum"],
        "dca_spd_sum": test_spd_summary["dca_spd_sum"],
        "extra_spd_sum_vs_dca": test_spd_summary["extra_spd_sum_vs_dca"],

        "strategy_spd_avg": test_spd_summary["strategy_spd_avg"],
        "dca_spd_avg": test_spd_summary["dca_spd_avg"],
        "extra_spd_avg_vs_dca": test_spd_summary["extra_spd_avg_vs_dca"],

        "spd_ratio": test_spd_summary["spd_ratio"],
        "improvement_pct": test_spd_summary["improvement_pct"],
        "result": get_status_from_pct_diff(test_spd_summary["improvement_pct"]),
    },
])

display(final_summary_df.round(6))


# ============================================================


,period,windows,wins,losses,ties,win_rate_pct
0,train,6,5,1,0,83.33
1,test,2,2,0,0,100.00


,period,start_date,end_date,windows,wins,losses,ties,win_rate_pct,exp_decay_percentile,uniform_exp_decay_percentile,...,extra_sats_vs_dca,strategy_spd_sum,dca_spd_sum,extra_spd_sum_vs_dca,strategy_spd_avg,dca_spd_avg,extra_spd_avg_vs_dca,spd_ratio,improvement_pct,result
0,train,2018-01-01,2023-12-31,6,5,1,0,83.333333,54.475708,54.475708,...,4.224665e+06,54769.121967,50544.456693,4224.665274,9128.186994,8424.076115,704.110879,1.083583,8.358316,better
1,test,2024-01-01,2025-12-31,2,2,0,0,100.000000,73.684211,73.684211,...,1.066452e+05,2693.976696,2587.331527,106.645169,1346.988348,1293.665763,53.322585,1.041218,4.121821,better


## Cell 14A: Strategy and allocation diagnostics

Show diagnostic checks for selected strategies and final allocation concentration after capping.

In [16]:
# Cell 14A: Strategy and allocation diagnostics
# ============================================================
# This cell checks which strategies are selected and whether
# the final allocations are too concentrated.

print("Best strategy mapping counts:")
display(best_mapping_df["strategy"].value_counts())

print("Selected strategy counts in train:")
display(train_strategy_df["selected_strategy"].value_counts())

print("Selected strategy counts in test:")
display(test_strategy_df["selected_strategy"].value_counts())

diagnostics_df = pd.DataFrame([
    {
        "period": "train",
        "mvrv_days_selected": int((train_strategy_df["selected_strategy"] == "stacksats_mvrv_weight").sum()),
        "max_weight": train_strategy_df["final_strategy_weight"].max(),
        "avg_weight": train_strategy_df["final_strategy_weight"].mean(),
        "days_above_dca": int(
            (train_strategy_df["final_strategy_weight"] > train_strategy_df["dca_weight"]).sum()
        ),
        "composite_days_selected": int((train_strategy_df["selected_strategy"] == "composite_weight").sum()),
    },
    {
        "period": "test",
        "mvrv_days_selected": int((test_strategy_df["selected_strategy"] == "stacksats_mvrv_weight").sum()),
        "max_weight": test_strategy_df["final_strategy_weight"].max(),
        "avg_weight": test_strategy_df["final_strategy_weight"].mean(),
        "days_above_dca": int(
            (test_strategy_df["final_strategy_weight"] > test_strategy_df["dca_weight"]).sum()
        ),
        "composite_days_selected": int((test_strategy_df["selected_strategy"] == "composite_weight").sum()),
    },
])

display(diagnostics_df.round(8))


Best strategy mapping counts:


strategy
composite_weight             3
stacksats_mvrv_weight        3
stacksats_momentum_weight    2
sma_180d_weight              2
Name: count, dtype: int64

Selected strategy counts in train:


selected_strategy
composite_weight             843
stacksats_momentum_weight    673
stacksats_mvrv_weight        568
sma_180d_weight              106
Name: count, dtype: int64

Selected strategy counts in test:


selected_strategy
stacksats_momentum_weight    410
stacksats_mvrv_weight        154
composite_weight             139
sma_180d_weight               27
Name: count, dtype: int64

,period,mvrv_days_selected,max_weight,avg_weight,days_above_dca,composite_days_selected
0,train,568,0.041096,0.00274,444,843
1,test,154,0.041096,0.00274,142,139


## Cell 15: Prepare plot data

Combine train and test daily outputs into a single dataframe for visualization.

In [17]:
# Cell 15: Prepare plot data
# ============================================================
# This cell combines train and test daily outputs for plotting.

train_plot_df = train_strategy_df.copy()
train_plot_df["split"] = "Train"

test_plot_df = test_strategy_df.copy()
test_plot_df["split"] = "Test"

combined_strategy_df = pd.concat([train_plot_df, test_plot_df], ignore_index=True)
combined_strategy_df["date"] = pd.to_datetime(combined_strategy_df["date"])

train_plot_start = pd.to_datetime(TRAIN_START)
test_plot_start = pd.to_datetime(TEST_START)
test_plot_end = pd.to_datetime(TEST_END)


# ============================================================


## Cell 16: Chart function

Define the final Plotly chart showing BTC price, strategy weights, DCA weights, and train/test performance annotations.

In [18]:
# Cell 16: Chart function
# ============================================================
# This cell defines the main BTC price vs strategy/DCA weight chart.
# It shows BTC price, final strategy weights, DCA weights, and train/test summary metrics.

def plot_train_test_period_with_spd_header(
    combined_df,
    train_summary,
    test_summary,
    title="BTC Price vs Simplified MVRV, Momentum, SMA, Composite Regime Framework and Uniform DCA Weights",
):
    """
    Build the final train/test chart.

    Parameters
    ----------
    combined_df : pd.DataFrame
        Daily train and test strategy output combined into one dataframe.
    train_summary : dict
        Summary metrics for the training period.
    test_summary : dict
        Summary metrics for the test period.
    title : str
        Chart title.

    Returns
    -------
    plotly.graph_objects.Figure
        Plotly figure showing BTC price, strategy weight, DCA weight,
        and train/test performance text.
    """
    df = combined_df.copy()
    df["date"] = pd.to_datetime(df["date"])

    train_arrow_text, train_arrow_color = format_arrow_text(
        train_summary["extra_spd_sum_vs_dca"],
        train_summary["improvement_pct"],
    )

    test_arrow_text, test_arrow_color = format_arrow_text(
        test_summary["extra_spd_sum_vs_dca"],
        test_summary["improvement_pct"],
    )

    tickvals, ticktext = build_log_tick_values_and_text()

    year_ticks = pd.date_range(
        start=pd.to_datetime(TRAIN_START),
        end=pd.to_datetime(TEST_END),
        freq="YS",
    )

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(
        go.Scatter(
            x=df["date"],
            y=df["price_usd"],
            mode="lines",
            name="BTC Price",
            line=dict(color="black", width=2.0),
            hovertemplate="<b>%{x|%Y-%m-%d}</b><br>Price: $%{y:,.0f}<extra></extra>",
        ),
        secondary_y=False,
    )

    fig.add_trace(
        go.Scatter(
            x=df["date"],
            y=df["final_strategy_weight"],
            mode="lines",
            name="Strategy Weight",
            line=dict(color="green", width=1.8),
            hovertemplate="<b>%{x|%Y-%m-%d}</b><br>Strategy Weight: %{y:.6f}<extra></extra>",
        ),
        secondary_y=True,
    )

    fig.add_trace(
        go.Scatter(
            x=df["date"],
            y=df["dca_weight"],
            mode="lines",
            name=f"Uniform DCA (1/{WINDOW_SIZE})",
            line=dict(color="red", width=1.4, dash="dash"),
            hovertemplate="<b>%{x|%Y-%m-%d}</b><br>DCA Weight: %{y:.6f}<extra></extra>",
        ),
        secondary_y=True,
    )

    fig.add_vrect(
        x0=test_plot_start,
        x1=test_plot_end,
        fillcolor="lightgreen",
        opacity=0.15,
        layer="below",
        line_width=0,
    )

    fig.add_vline(
        x=test_plot_start,
        line_width=1.2,
        line_dash="dot",
        line_color="gray",
    )

    fig.add_annotation(
        x=test_plot_start + pd.Timedelta(days=40),
        y=0.965,
        xref="x",
        yref="paper",
        text="<b>Test</b>",
        showarrow=False,
        font=dict(size=18, color="black", family="Arial Black"),
        xanchor="left",
    )

    fig.update_yaxes(
        title_text="<b>BTC Price (USD, log scale)</b>",
        type="log",
        tickmode="array",
        tickvals=tickvals,
        ticktext=ticktext,
        secondary_y=False,
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=False,
        title_font=dict(size=20, color="black", family="Arial Black"),
        tickfont=dict(size=13, color="black", family="Arial Black"),
    )

    fig.update_yaxes(
        title_text="<b>Allocation Weight</b>",
        secondary_y=True,
        showgrid=False,
        range=[0, 0.082],
        zeroline=False,
        title_font=dict(size=20, color="black", family="Arial Black"),
        tickfont=dict(size=13, color="black", family="Arial Black"),
    )

    fig.update_xaxes(
        title_text="<b>Year</b>",
        range=[train_plot_start, test_plot_end],
        tickmode="array",
        tickvals=year_ticks,
        tickformat="%Y",
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=False,
        title_font=dict(size=20, color="black", family="Arial Black"),
        tickfont=dict(size=14, color="black", family="Arial Black"),
    )

    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b>",
            x=0.5,
            xanchor="center",
            y=0.985,
            font=dict(size=30, color="black", family="Arial Black"),
        ),
        template="plotly_white",
        hovermode="x unified",
        width=1900,
        height=1050,
        showlegend=False,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=35, r=35, t=120, b=35),
    )

    fig.add_annotation(
        x=0.5,
        y=1.095,
        xref="paper",
        yref="paper",
        showarrow=False,
        align="center",
        text=(
            f"<b>Train — Strategy: {train_summary['strategy_spd_sum']:,.2f} sats/$ | "
            f"DCA: {train_summary['dca_spd_sum']:,.2f} sats/$ | "
            f"Excess: <span style='color:{train_arrow_color};'>{train_arrow_text}</span> | "
            f"Win Rate: {train_summary['win_rate_pct']:.1f}% | "
            f"ExpDecayPct: {train_summary['exp_decay_percentile']:.2f}</b>"
        ),
        font=dict(size=17, color="black", family="Arial Black"),
    )

    fig.add_annotation(
        x=0.5,
        y=1.055,
        xref="paper",
        yref="paper",
        showarrow=False,
        align="center",
        text=(
            f"<b>Test — Strategy: {test_summary['strategy_spd_sum']:,.2f} sats/$ | "
            f"DCA: {test_summary['dca_spd_sum']:,.2f} sats/$ | "
            f"Excess: <span style='color:{test_arrow_color};'>{test_arrow_text}</span> | "
            f"Win Rate: {test_summary['win_rate_pct']:.1f}% | "
            f"ExpDecayPct: {test_summary['exp_decay_percentile']:.2f}</b>"
        ),
        font=dict(size=17, color="black", family="Arial Black"),
    )

    return fig


# ============================================================


## Cell 17: Show chart

Render the final chart.

In [19]:
# Cell 17: Show chart
# ============================================================
# This cell renders the final chart.

main_fig = plot_train_test_period_with_spd_header(
    combined_strategy_df,
    train_spd_summary,
    test_spd_summary,
    title="BTC Price vs Simplified MVRV, Momentum, SMA, Composite Regime Framework and Uniform DCA Weights",
)

main_fig.show()
